In [1]:
import subprocess
subprocess.run(['pip', 'install', 'ase', 'pymatgen', '--break-system-packages', '-q'])

CompletedProcess(args=['pip', 'install', 'ase', 'pymatgen', '--break-system-packages', '-q'], returncode=0)

In [2]:
from ase.build import surface
from ase.io import read
from ase.visualize.plot import plot_atoms
import matplotlib.pyplot as plt
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
import os

In [26]:
from ase.build import surface, make_supercell
from ase.io import read
from ase.visualize import view

alloy = read('bulk_structure/MgZn2_relaxed.cif')
slab = surface(alloy, (1, 0, 1), 4, vacuum=8)

matrix = [[2, 0, 0],
          [0, 2, 0],
          [0, 0, 1]]
slab_doubled = make_supercell(slab, matrix)

print("Atom index | Species | Position (x, y, z)")
for i, atom in enumerate(slab_doubled):
    print(f"  {i:3d} | {atom.symbol:2s} | "
          f"x={atom.position[0]:.4f} "
          f"y={atom.position[1]:.4f} "
          f"z={atom.position[2]:.4f}")

view(slab_doubled)

Atom index | Species | Position (x, y, z)
    0 | Mg | x=8.3622 y=3.2443 z=10.8117
    1 | Mg | x=6.0360 y=3.8755 z=8.6056
    2 | Mg | x=3.2694 y=4.6263 z=10.1320
    3 | Mg | x=0.9432 y=2.4935 z=9.2853
    4 | Zn | x=6.5005 y=0.9855 z=9.7086
    5 | Zn | x=2.8048 y=1.9883 z=11.7477
    6 | Zn | x=8.3484 y=1.8852 z=8.0000
    7 | Zn | x=9.0373 y=4.4239 z=8.0000
    8 | Zn | x=9.0566 y=0.2919 z=10.0673
    9 | Zn | x=4.6527 y=0.0858 z=11.4173
   10 | Zn | x=5.3610 y=2.6959 z=11.4173
   11 | Zn | x=3.9444 y=1.6791 z=9.3500
   12 | Mg | x=9.7594 y=0.1012 z=14.8898
   13 | Mg | x=7.4332 y=0.7324 z=12.6837
   14 | Mg | x=4.6665 y=1.4832 z=14.2102
   15 | Mg | x=3.7375 y=4.4992 z=13.3634
   16 | Zn | x=9.2949 y=2.9912 z=13.7868
   17 | Zn | x=5.5992 y=3.9940 z=15.8258
   18 | Zn | x=11.1427 y=3.8909 z=12.0781
   19 | Zn | x=10.4344 y=1.2808 z=12.0781
   20 | Zn | x=1.6653 y=2.2976 z=14.1454
   21 | Zn | x=7.4470 y=2.0915 z=15.4954
   22 | Zn | x=8.1553 y=4.7016 z=15.4954
   23 | Zn | x=6.73

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [16]:
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
import os

alloy = read('bulk_structure/MgZn2_relaxed.cif')
slab = surface(alloy, (1, 0, 1), 6, vacuum=8)

matrix = [[1, 0, 0],
          [0, 1, 0],
          [0, 0, 1]]
slab_doubled = make_supercell(slab, matrix)

# Move atom 89 to bottom surface
slab_doubled[69].position = [4.653, 0.0086, 31.807]


# Visualise to confirm
view(slab_doubled)

# Convert to pymatgen and write .data file
adaptor = AseAtomsAdaptor()
pymatgen_slab = adaptor.get_structure(slab_doubled)

os.makedirs("slabs", exist_ok=True)
lammps_data = LammpsData.from_structure(pymatgen_slab, atom_style="atomic")
lammps_data.write_file("slabs/mgzn2_101_6layers_reconstructed_ase.data")

mg_count = sum(1 for site in pymatgen_slab if site.species_string == "Mg")
zn_count = sum(1 for site in pymatgen_slab if site.species_string == "Zn")
print(f"Mg: {mg_count} Zn: {zn_count}")
print(f"Stoichiometric: {'YES' if zn_count == 2 * mg_count else 'NO'}")

Mg: 24 Zn: 48
Stoichiometric: YES


In [17]:
from pymatgen.core.surface import Slab
from pymatgen.io.lammps.data import LammpsData

lammps_data = LammpsData.from_file(
    "slabs/mgzn2_101_6layers_reconstructed_ase.data",
    atom_style="atomic"
)
structure = lammps_data.structure

# Convert to Slab object
slab = Slab(
    structure.lattice,
    structure.species,
    structure.frac_coords,
    miller_index=(1, 0, 1),
    oriented_unit_cell=structure,
    shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]]
)

print(f"Symmetric: {slab.is_symmetric()}")

Symmetric: False
